In [27]:
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
import ast
import os

In [28]:
# Load preprocessed dataset
dataset_file = np.load('final_augmented_dataset.npz')

X_train = dataset_file['X_train']
y_train = dataset_file['y_train']
X_val = dataset_file['X_val']
y_val = dataset_file['y_val']
X_test = dataset_file['X_test']
y_test = dataset_file['y_test']

# Verify Shapes
print(X_train.shape)
print(y_train.shape)

(22501, 5000, 1)
(22501, 3)


In [29]:
# Set parameters for model
initial_filters = 32
filters = 64
kernel_size = 7
pool_size = 2
units = 128
final_units = 3

# Build layers for model
model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(initial_filters, kernel_size, activation='relu', input_shape=(X_train.shape[1], 1)),
    tf.keras.layers.MaxPooling1D(pool_size),
    tf.keras.layers.Conv1D(filters, kernel_size, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size),
    tf.keras.layers.Conv1D(filters, kernel_size, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(units, activation='relu'),
    tf.keras.layers.Dense(final_units, activation='softmax')
])

# Verify model summary
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_18 (Conv1D)              │ (None, 4994, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_18 (MaxPooling1D) │ (None, 2497, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 2491, 64)       │        14,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_19 (MaxPooling1D) │ (None, 1245, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_20 (Conv1D)              │ (None, 1239, 64)       │        28,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_20 (MaxPooling1D) │ (None, 619, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,099 (203.51 KB)

 Trainable params: 52,099 (203.51 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
# Compile the model
model.compile('adam', 'categorical_crossentropy', metrics=['accuracy'])